# Simulación del estómago — Fase 2: transporte 1D/2D (advección–difusión–reacción)

**Qué añade esta fase.** Hasta ahora el estómago era un vaso perfectamente mezclado (modelo 0D). Aquí las concentraciones pasan a depender del **espacio**:
- El ácido entra por las **paredes** del cuerpo gástrico, no "en todo el volumen".
- La comida empieza como un **bolo** localizado.
- Un **campo de velocidades** mueve y mezcla el contenido.

La química es la de la fase 1b (tampón en equilibrio y activación de la pepsina) y entra como término de reacción en cada celda.

**Qué es y qué no es el flujo de esta fase.** Usamos un flujo **cinemático sustituto**: dos patrones de vórtice que se alternan unas 3 veces por minuto, como las contracciones antrales. No sale de resolver Navier–Stokes; esa será la fase 3. Lo que importa aquí es tener un transporte **verificado**, capaz de mover especies químicas con cualquier campo de velocidades que le demos después.

**Estructura del cuaderno**
1. Ecuaciones y discretización (volúmenes finitos).
2. Código del solucionador.
3. Verificación: órdenes de convergencia en 1D y 2D (solución manufacturada), conservación, positividad, pH vectorizado y regresión al modelo 0D.
4. Modelo 1D: dispersión axial del ácido.
5. Modelo 2D: un bolo de carne en el estómago, con y sin flujo, comparado con el modelo 0D.
6. Error numérico del experimento (convergencia de malla con Richardson) y sensibilidad al cierre de mezcla.

**Uso en Colab**: `Ejecutar todo`. Tarda unos 5–6 minutos en total. La celda de convergencia de malla es la más lenta (≈ 2 min).

## 1. Ecuaciones

### 1.1 Ecuación de advección–difusión–reacción
Para cada especie $c_i(\mathbf{x}, t)$:

$$ \boxed{\ \frac{\partial c_i}{\partial t} + \nabla\!\cdot(\mathbf{u}\, c_i) = \nabla\!\cdot\!\big(D_i \nabla c_i\big) + R_i(\mathbf{c}) + s_i(\mathbf{x})\ } $$

| Término | Significado físico | En el código |
|---|---|---|
| $\nabla\cdot(\mathbf{u} c_i)$ | El flujo arrastra la especie (advección) | `operador_adveccion` |
| $\nabla\cdot(D_i\nabla c_i)$ | Difusión molecular más mezcla no resuelta | `operador_difusion` |
| $R_i$ | Reacciones químicas de la fase 1b | `reacciones` |
| $s_i$ | Secreción de H⁺ y pepsinógeno desde la pared | `fuente_unit` |

**Incompresibilidad.** El contenido gástrico es prácticamente incompresible: $\nabla\cdot\mathbf{u} = 0$. Por eso escribimos $\mathbf{u}$ a partir de una **función de corriente** $\psi$:
$$ u = \frac{\partial \psi}{\partial y}, \qquad v = -\frac{\partial \psi}{\partial x} \quad\Longrightarrow\quad \nabla\cdot\mathbf{u} = 0 . $$
Si $\psi = 0$ en el contorno, el flujo no atraviesa las paredes.

**Condiciones de contorno.** Las paredes son impermeables al flujo y a la difusión, es decir, tienen flujo nulo (condición de Neumann). La secreción de la pared entra como fuente en las celdas pegadas a ella. En esta fase el volumen es fijo: no hay vaciado. El vaciado se acoplará con el flujo en la fase 3.

### 1.2 Dos fases: comida sólida y líquido gástrico
- **La comida es un sólido inmóvil** en esta fase. Su tampón ($B_t$) y su proteína ($S$) no se advectan; solo tienen una difusión molecular despreciable. Modelamos el bolo de carne masticada como una masa **porosa**: el jugo gástrico la empapa y la atraviesa. Moverla y romperla será trabajo de la fase de partículas.
- **El líquido se mueve**: H⁺ libre, pepsinógeno, pepsina y péptidos se advectan con el flujo y se mezclan con $D_{mix}$.
- **Protones.** El total $T$ es la suma de los libres ($h$, que viajan con el líquido) y los ligados a la comida ($T-h$, que se quedan con ella). Por eso solo se advecta $h$, y el flujo difusivo es
$$ \mathbf{J}_T = -(D_{H} + D_{mix})\nabla h - D_{food}\nabla (T - h). $$
En cada paso, $h$ se obtiene del equilibrio químico (fase 1b), resolviendo en todas las celdas a la vez con un Newton vectorizado.

### 1.3 Mezcla no resuelta: el parámetro de cierre $D_{mix}$
La difusión molecular del H⁺ ($\approx 10^{-5}$ cm²/s) es tan lenta que mezclaría 1 cm en más de un día. Como el bolo es poroso, $D_{mix}$ actúa también **dentro** de él y controla lo deprisa que el ácido penetra en la comida. En el estómago real, la mezcla la hacen el flujo y estructuras más pequeñas que nuestra malla (chorros, remolinos, movimiento de partículas). Esas estructuras se representan con una **difusividad efectiva** $D_{mix}$, que se suma a la molecular.

Es un **parámetro de cierre**: no se mide directamente. La fase 3, con Navier–Stokes, servirá para estimarlo. La sección 6 muestra por qué hace falta: sin él, la "mezcla" que se obtiene la decide la malla (difusión numérica), no la física.

### 1.4 Números adimensionales que gobiernan el problema
$$ \mathrm{Pe} = \frac{U L}{D_{mix}} \ (\text{advección/difusión}), \qquad \mathrm{Pe}_{celda} = \frac{U\,\Delta x}{D_{mix}}, \qquad \mathrm{Da} = \frac{k_{reac}\, L}{U}\ (\text{reacción/transporte}). $$
Para que la malla resuelva la difusión hace falta que $\mathrm{Pe}_{celda}$ sea de orden 1–10. Si es mucho mayor, la difusión numérica del esquema supera a la física.

### 1.5 Discretización
- **Volúmenes finitos**: cada celda guarda su valor medio y los flujos se calculan en las caras. Lo que sale de una celda entra en la vecina, así que la masa se conserva **exactamente** (salvo redondeo).
- **Advección**: reconstrucción MUSCL con limitador de **van Leer**. Es de 2º orden donde la solución es suave y no genera oscilaciones ni valores negativos en los frentes: es un esquema TVD. Como comparación incluimos *upwind* (1er orden) y *minmod*.
- **Difusión**: diferencias centradas de 2º orden.
- **Velocidades**: $\psi$ se evalúa en los vértices de la malla y $u$, $v$ en las caras. Así la divergencia discreta es **exactamente cero**.
- **Tiempo**: Heun (SSP-RK2), que conserva las propiedades TVD y de positividad. El paso cumple $\mathrm{CFL} = U\Delta t/\Delta x \le 0.4$ y $D\Delta t/\Delta x^2 \le 0.125$, con un tope de 0.01 min por la química. El límite de difusión es más estricto que el clásico 0.25 porque, en 2D y combinado con la advección, Heun necesita margen: con 0.2 el caso de $D_{mix}=2$ se vuelve inestable.

## 2. Código

In [ ]:
import math, time
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
COLOR_ESQ = {"upwind": "#2a78d6", "minmod": "#eb6834", "vanleer": "#1baf7a"}
MARCA_ESQ = {"upwind": "o", "minmod": "s", "vanleer": "^"}
COLOR_MOD = {"0D (mezcla perfecta)": GRIS, "2D en reposo": "#2a78d6", "2D con vórtices": "#eb6834"}
CMAP_PH = "viridis"            # secuencial: pH bajo (ácido) = oscuro
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
VERIFICACION = []

### 2.1 Malla, velocidades y operadores de transporte

In [ ]:
@dataclass
class Malla:
    # Malla cartesiana de volúmenes finitos. ny = 1 da un problema 1D.
    nx: int
    ny: int
    Lx: float
    Ly: float
    periodica: bool = False      # periódica en x (solo para tests); si no, paredes en todo el contorno

    def __post_init__(self):
        self.dx = self.Lx / self.nx
        self.dy = self.Ly / self.ny
        self.xc = (np.arange(self.nx) + 0.5) * self.dx          # centros de celda
        self.yc = (np.arange(self.ny) + 0.5) * self.dy
        self.X, self.Y = np.meshgrid(self.xc, self.yc)           # forma (ny, nx)
        self.xf = np.arange(self.nx + 1) * self.dx              # caras
        self.yf = np.arange(self.ny + 1) * self.dy

    @property
    def area_celda(self):
        return self.dx * self.dy


# ---------------------------------------------------------------------------
# Campos de velocidad discretamente libres de divergencia (a partir de psi en los vértices)
# ---------------------------------------------------------------------------
def velocidades_desde_psi(psi_v, m):
    # psi_v: función de corriente en los vértices, forma (ny+1, nx+1).
    # u en caras x (ny, nx+1) = dpsi/dy ; v en caras y (ny+1, nx) = -dpsi/dx.
    # Con esta construcción, la divergencia discreta de cada celda es exactamente 0.
    u = (psi_v[1:, :] - psi_v[:-1, :]) / m.dy
    v = -(psi_v[:, 1:] - psi_v[:, :-1]) / m.dx
    return u, v


def divergencia_discreta(u, v, m):
    return (u[:, 1:] - u[:, :-1]) / m.dx + (v[1:, :] - v[:-1, :]) / m.dy


# ---------------------------------------------------------------------------
# Advección: volúmenes finitos con reconstrucción MUSCL y limitador
# ---------------------------------------------------------------------------
def limitador(a, b, tipo):
    # Pendiente limitada a partir de las diferencias izquierda (a) y derecha (b)
    if tipo == "upwind":
        return np.zeros_like(a)
    if tipo == "vanleer":
        ab = a * b
        den = np.where(ab > 0, a + b, 1.0)          # evita dividir por cero donde la pendiente es nula
        return np.where(ab > 0, 2 * ab / den, 0.0)
    if tipo == "minmod":
        return np.where(a * b > 0, np.sign(a) * np.minimum(np.abs(a), np.abs(b)), 0.0)
    raise ValueError(tipo)


def flujo_advectivo_1eje(c, vel, eje, esquema, periodica):
    # Flujo numérico F = vel·c_cara en todas las caras perpendiculares a `eje` (-1: x, -2: y).
    # c: (..., ny, nx). vel: velocidad en las caras de ese eje, forma (ny, nx+1) o (ny+1, nx).
    cc = np.moveaxis(c, eje, -1)                     # el eje de trabajo pasa al final
    vv = np.moveaxis(vel, eje, -1)
    n = cc.shape[-1]
    modo = "wrap" if periodica else "edge"
    pad = [(0, 0)] * (cc.ndim - 1) + [(2, 2)]
    g = np.pad(cc, pad, mode=modo)                   # 2 celdas fantasma por lado
    d = np.diff(g, axis=-1)                          # diferencias entre celdas consecutivas
    s = limitador(d[..., :-1], d[..., 1:], esquema)  # pendiente en celdas -1 .. n
    gc = g[..., 1:-1]                                # celdas -1 .. n
    izq = gc[..., :-1] + 0.5 * s[..., :-1]           # valor en la cara desde la celda izquierda
    der = gc[..., 1:] - 0.5 * s[..., 1:]             # valor en la cara desde la celda derecha
    F = np.where(vv >= 0, vv * izq, vv * der)        # caras 0 .. n  (n+1 caras)
    if not periodica:
        F[..., 0] = 0.0
        F[..., -1] = 0.0                             # paredes impermeables
    return np.moveaxis(F, -1, eje)


def flujo_difusivo_1eje(c, D, h, eje, periodica):
    cc = np.moveaxis(c, eje, -1)
    if periodica:
        g = np.concatenate([cc[..., -1:], cc, cc[..., :1]], axis=-1)
        F = -D * np.diff(g, axis=-1) / h
        F = F[..., :]                                # n+1 caras (la primera y la última coinciden)
    else:
        F = np.zeros(cc.shape[:-1] + (cc.shape[-1] + 1,))
        F[..., 1:-1] = -D * np.diff(cc, axis=-1) / h # pared: flujo difusivo nulo (Neumann)
    return np.moveaxis(F, -1, eje)


def operador_adveccion(c, u, v, m, esquema="vanleer"):
    # -div(u c) para uno o varios campos c (..., ny, nx); u, v en las caras
    Fx = flujo_advectivo_1eje(c, u, -1, esquema, m.periodica)
    out = -(Fx[..., 1:] - Fx[..., :-1]) / m.dx
    if m.ny > 1:
        Fy = flujo_advectivo_1eje(c, v, -2, esquema, False)
        out -= (Fy[..., 1:, :] - Fy[..., :-1, :]) / m.dy
    return out


def operador_difusion(c, D, m):
    # div(D grad c) con D escalar o un valor por campo (forma (k,1,1)); paredes sin flujo
    Fx = flujo_difusivo_1eje(c, D, m.dx, -1, m.periodica)
    out = -(Fx[..., 1:] - Fx[..., :-1]) / m.dx
    if m.ny > 1:
        Fy = flujo_difusivo_1eje(c, D, m.dy, -2, False)
        out -= (Fy[..., 1:, :] - Fy[..., :-1, :]) / m.dy
    return out


def operador_transporte(c, u, v, D, m, esquema="vanleer"):
    return operador_adveccion(c, u, v, m, esquema) + operador_difusion(c, D, m)


def paso_ssprk2(c, rhs, t, dt):
    # Heun = SSP-RK2: conserva las propiedades TVD/positividad del paso de Euler con CFL <= 1
    c1 = c + dt * rhs(t, c)
    return 0.5 * (c + c1 + dt * rhs(t + dt, c1))


def dt_estable(m, umax, Dmax, cfl=0.4, fo=0.125, dt_max=np.inf):
    lim = [dt_max]
    if umax > 0:
        lim.append(cfl * min(m.dx, m.dy if m.ny > 1 else np.inf) / umax)
    if Dmax > 0:
        lim.append(fo * min(m.dx, m.dy if m.ny > 1 else np.inf) ** 2 / Dmax)
    return min(lim)

### 2.2 Química (fase 1b) y estómago 2D

In [ ]:
@dataclass
class ParamQ:
    # Química de la fase 1b (versión de tampón en equilibrio). Tiempo en min.
    k_a1: float = 1.7e-3
    k_a2: float = 0.83
    k_cat: float = 15.0
    K_M: float = 5.0
    pH_opt: float = 2.0
    sigma: float = 1.5
    k_H: float = 1e-5
    pKa: tuple = (2.5, 3.5, 4.5, 5.5, 6.5)
    beta_prot: float = 0.30      # mmol/(g proteína · pH)
    f_pep: float = 0.30
    r_acid: float = 0.375        # mmol/min  secreción total de H+
    r_G: float = 1.0e-4          # mmol/min  secreción total de pepsinógeno


@dataclass
class ParamT:
    # Transporte. Longitudes en cm, tiempo en min.
    U: float = 15.0              # cm/min  velocidad máxima (0.25 cm/s, del orden de la onda peristáltica)
    T_w: float = 20 / 60         # min     periodo de alternancia de vórtices (3 por minuto)
    D_H: float = 5.6e-3          # cm²/min difusión molecular de H+ (9.3e-5 cm²/s)
    D_enz: float = 6e-5          # cm²/min pepsina / pepsinógeno (~1e-6 cm²/s)
    D_food: float = 6e-6         # cm²/min proteína y tampón ligados a la comida (~1e-7 cm²/s)
    D_pep: float = 3e-4          # cm²/min péptidos (~5e-6 cm²/s)
    D_mix: float = 0.5           # cm²/min difusividad de mezcla no resuelta (cierre; supuesto)
    W: float = 5.0               # cm      profundidad de la losa 2D (volumen = Lx·Ly·W)
    x_sec: float = 0.6           # —       fracción proximal de las paredes que secreta
    esquema: str = "vanleer"
    secrecion: str = "pared"     # 'pared' o 'uniforme' (test de regresión)
    flujo: str = "vortices"      # 'vortices' o 'reposo'


ESPECIES = ["T", "Bt", "G", "E", "S", "P"]   # orden de las filas del array de estado


def resolver_h(T, Bt, pKa, h0=None, tol=1e-13, max_iter=60):
    # Newton vectorizado para h + sum_j (Bt/n)·h/(h+Ka_j) = T  (todas las celdas a la vez)
    n = len(pKa)
    Ka = [10 ** (3.0 - pk) for pk in pKa]
    Bj = Bt / n
    h = np.clip(T if h0 is None else h0, 1e-12, None)
    h = np.minimum(h, np.maximum(T, 1e-12))
    for _ in range(max_iter):
        F = h - T
        dF = np.ones_like(h)
        for K in Ka:
            F = F + Bj * h / (h + K)
            dF = dF + Bj * K / (h + K) ** 2
        hn = h - F / dF
        hn = np.where(hn <= 0, 0.1 * h, hn)
        if np.max(np.abs(hn - h) / hn) < tol:
            return hn
        h = hn
    return h


iT, iB, iG, iE, iS, iP = range(6)


def reacciones(C, h, q):
    # Velocidades de reacción por celda (mM/min y g/L/min), idénticas a la fase 1b.
    # C: (6, ny, nx). Devuelve un array de la misma forma.
    pH = 3.0 - np.log10(h)
    phi = np.exp(-((pH - q.pH_opt) / q.sigma) ** 2)
    G, E, S = C[iG], C[iE], C[iS]
    r_a = (q.k_a1 * h + q.k_a2 * h * E) * G
    r_e = q.k_cat * E * phi * S / (q.K_M + S)
    r_h = q.k_H * h * S
    R = np.zeros_like(C)
    R[iG], R[iE] = -r_a, r_a
    R[iS], R[iP] = -(r_e + r_h), r_e + r_h
    return R                       # T y Bt no cambian por reacción (el equilibrio solo redistribuye)


class Estomago2D:
    # Advección–difusión–reacción en una losa 2D de profundidad W.
    def __init__(self, m, q=None, pt=None):
        self.m, self.q, self.pt = m, q or ParamQ(), pt or ParamT()
        self.vol_celda_mL = m.dx * m.dy * self.pt.W           # cm³ = mL
        self.V_total = m.Lx * m.Ly * self.pt.W
        # velocidades de los dos modos de vórtice: psi en los vértices -> caras (divergencia nula exacta)
        Xv, Yv = np.meshgrid(m.xf, m.yf)
        self.modos = []
        for nmodo in (1, 2):
            psi = np.sin(nmodo * np.pi * Xv / m.Lx) * np.sin(np.pi * Yv / m.Ly)
            u, v = velocidades_desde_psi(psi, m)
            esc = max(np.abs(u).max(), np.abs(v).max())
            self.modos.append((u / esc, v / esc))
        # Fases: la comida (tampón Bt y proteína S) es un SÓLIDO inmóvil en esta fase; el líquido
        # (H+ libre, enzimas, péptidos) se mueve con el flujo y se mezcla con D_mix.
        pt = self.pt
        self.movil = np.array([1.0, 0.0, 1.0, 1.0, 0.0, 1.0])[:, None, None]
        Dmol = np.array([0.0, pt.D_food, pt.D_enz, pt.D_enz, pt.D_food, pt.D_pep])
        self.D = (Dmol + pt.D_mix * self.movil[:, 0, 0])[:, None, None]
        # fuente de secreción (mM/min por celda) para una tasa total de 1 mmol/min
        mask = np.zeros((m.ny, m.nx))
        prox = m.xc < pt.x_sec * m.Lx
        if m.ny > 1:
            mask[0, prox] = 1.0
            mask[-1, prox] = 1.0
        if pt.secrecion == "uniforme":
            self.fuente_unit = np.full((m.ny, m.nx), 1000.0 / self.V_total)
        elif m.ny == 1:                                          # 1D: el tramo proximal secreta en volumen
            seg = prox[None, :].astype(float)
            self.fuente_unit = seg / (seg.sum() * self.vol_celda_mL) * 1000.0
        else:
            area = mask.sum() * m.dx * pt.W                     # cm² de pared secretora
            self.fuente_unit = mask / area / m.dy * 1000.0       # flujo/(espesor de celda) -> mM/min
        self.mask_sec = mask
        self._h = None

    def velocidad(self, t):
        if self.pt.flujo == "reposo":
            return 0.0 * self.modos[0][0], 0.0 * self.modos[0][1]
        w = 0.5 * (1 + np.sin(2 * np.pi * t / self.pt.T_w))
        u = self.pt.U * (w * self.modos[0][0] + (1 - w) * self.modos[1][0])
        v = self.pt.U * (w * self.modos[0][1] + (1 - w) * self.modos[1][1])
        return u, v

    def h_de(self, C):
        self._h = resolver_h(C[iT], C[iB], self.q.pKa, self._h)
        return self._h

    def rhs(self, t, C):
        m, pt = self.m, self.pt
        u, v = self.velocidad(t)
        h = resolver_h(C[iT], C[iB], self.q.pKa, self._h)
        # advección: solo la fase líquida. Para T se advecta la parte libre h (los ligados van con la comida)
        A = C * self.movil
        A[iT] = h
        out = operador_adveccion(A, u, v, m, pt.esquema)
        # difusión: libres con D_H + D_mix, ligados a la comida con D_food
        dif = operador_difusion(C, self.D, m)
        dif[iT] = operador_difusion(h, pt.D_H + pt.D_mix, m) + operador_difusion(C[iT] - h, pt.D_food, m)
        out += dif + reacciones(C, h, self.q)
        out[iT] += self.q.r_acid * self.fuente_unit
        out[iG] += self.q.r_G * self.fuente_unit
        return out

    def dt(self, cfl=0.4, fo=0.125, dt_max=0.01):
        umax = self.pt.U if self.pt.flujo != "reposo" else 0.0
        return dt_estable(self.m, umax, float(self.D.max()) + self.pt.D_H, cfl, fo, dt_max)

    def integrar(self, C0, t_final, dt=None, cada=1.0):
        dt = dt or self.dt()
        n = int(math.ceil(t_final / dt - 1e-9)); dt = t_final / n
        C = C0.copy()
        self._h = None
        guardar_cada = max(1, int(round(cada / dt)))
        tiempos, campos, hs = [], [], []
        for i in range(n + 1):
            t = i * dt
            h = self.h_de(C)
            if i % guardar_cada == 0 or i == n:
                tiempos.append(t); campos.append(C.copy()); hs.append(h.copy())
            if i == n:
                break
            C = paso_ssprk2(C, self.rhs, t, dt)
        return Historia(np.array(tiempos), np.array(campos), np.array(hs), self, dt)

    def total(self, campo):        # cantidad total: mmol si el campo está en mM, g si está en g/L
        return float(np.sum(campo) * self.vol_celda_mL / 1000.0)


@dataclass
class Historia:
    t: np.ndarray
    C: np.ndarray        # (nt, 6, ny, nx)
    h: np.ndarray        # (nt, ny, nx)
    sim: "Estomago2D"
    dt: float

    @property
    def pH(self):
        return 3.0 - np.log10(self.h)

    def totales(self, especie):
        return self.C[:, especie].sum(axis=(1, 2)) * self.sim.vol_celda_mL / 1000.0

    def pH_alimento(self):
        w = self.C[:, iS] + self.C[:, iP]
        return (w * self.pH).sum(axis=(1, 2)) / w.sum(axis=(1, 2))

    def pH_liquido(self):
        # pH medio del líquido: celdas fuera de la comida inicial (proteína < 1 % del máximo)
        mask = self.C[0, iS] < 0.01 * self.C[0, iS].max()
        return self.pH[:, mask].mean(axis=1)

    def dispersion_pH(self):
        return self.pH.reshape(len(self.t), -1).std(axis=1)

    def frac_hidrolizada(self, proteina_g):
        return self.totales(iP) / proteina_g

In [ ]:
def estado_bolo(sim, proteina_g=14.0, pH0=6.0, centro=(3.0, 2.0), radio=1.5, delta=0.2,
                V_basal=25.0, pH_basal=1.5):
    # Bolo de comida circular (borde suave) en un estómago lleno de agua + jugo basal
    m, q = sim.m, sim.q
    r = np.hypot(m.X - centro[0], m.Y - centro[1])
    chi = 0.5 * (1 - np.tanh((r - radio) / delta))
    dens = chi / (chi.sum() * sim.vol_celda_mL)               # 1/mL, integra 1
    n = len(q.pKa)
    Bt_tot = q.beta_prot * proteina_g * (max(q.pKa) - min(q.pKa) + 1)   # mmol
    C = np.zeros((6, m.ny, m.nx))
    C[iB] = Bt_tot * dens * 1000.0                             # mM
    C[iS] = q.f_pep * proteina_g * dens * 1000.0               # g/L
    C[iE] = 0.04 * V_basal / sim.V_total                       # pepsina del jugo basal (mM)
    h0 = 10 ** (3 - pH0)
    ligados = sum((C[iB] / n) / (1 + 10 ** (pH0 - pk)) for pk in q.pKa)
    H_basal = 10 ** (3 - pH_basal) * V_basal / sim.V_total     # mM, repartido uniforme
    C[iT] = ligados + h0 + H_basal
    return C

## 3. Verificación

### T1 — Advección 1D (periódica) contra la solución exacta
Una onda $c = 1 + a\sin(kx)$ transportada a velocidad $U$ conserva su forma: $c(x,t) = 1 + a\sin(k(x - Ut))$. Con el paso de tiempo proporcional a $\Delta x$ (CFL fijo), el error mide el orden conjunto en espacio y tiempo.

In [ ]:
def caso_1d(nx, esquema, D=0.0, T=1.0, U=1.0, a=0.5, L=1.0):
    m = Malla(nx, 1, L, 1.0, periodica=True)
    u = np.full((1, nx + 1), U); v = np.zeros((2, nx))
    k = 2 * np.pi / L
    rhs = lambda t, c: operador_transporte(c, u, v, D, m, esquema)
    dt = dt_estable(m, U, D, cfl=0.4); n = int(math.ceil(T / dt)); dt = T / n
    c = 1 + a * np.sin(k * m.X)
    for i in range(n):
        c = paso_ssprk2(c, rhs, i * dt, dt)
    exacta = 1 + a * np.sin(k * (m.X - U * T)) * np.exp(-D * k * k * T)
    return m, c, exacta, float(np.mean(np.abs(c - exacta)))

ns = [32, 64, 128, 256, 512]
res_T1 = {esq: [caso_1d(n, esq)[3] for n in ns] for esq in COLOR_ESQ}
res_T2 = [caso_1d(n, "vanleer", D=0.01)[3] for n in ns[:4]]

def orden(errs, ns_):
    return [math.log(errs[i] / errs[i + 1]) / math.log(ns_[i + 1] / ns_[i]) for i in range(len(errs) - 1)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
m_, c_, ex_, _ = caso_1d(32, "upwind")
a1.plot(m_.xc, ex_[0], color=GRIS, lw=3, alpha=0.4, label="exacta")
for esq in COLOR_ESQ:
    m_, c_, ex_, _ = caso_1d(32, esq)
    a1.plot(m_.xc, c_[0], marker=MARCA_ESQ[esq], ms=4, mfc="none", lw=1, color=COLOR_ESQ[esq], label=esq)
a1.set(xlabel="x", ylabel="c", title="Tras 1 vuelta completa, 32 celdas"); a1.legend(fontsize=9)
for esq in COLOR_ESQ:
    p = np.mean(orden(res_T1[esq], ns)[-2:])
    a2.loglog(ns, res_T1[esq], marker=MARCA_ESQ[esq], color=COLOR_ESQ[esq], label=f"{esq} (p ≈ {p:.2f})")
p2 = np.mean(orden(res_T2, ns[:4])[-2:])
a2.loglog(ns[:4], res_T2, ls="--", marker="^", color=COLOR_ESQ["vanleer"], mfc="none",
          label=f"vanleer + difusión (p ≈ {p2:.2f})")
a2.minorticks_off(); a2.set_xticks(ns); a2.set_xticklabels([str(n) for n in ns])
a2.set(xlabel="nº de celdas", ylabel="error L1", title="Convergencia (CFL fijo)"); a2.legend(fontsize=9)
fig.suptitle("T1–T2 · Advección y advección–difusión 1D", fontweight="bold"); fig.tight_layout(); plt.show()

p_up = np.mean(orden(res_T1["upwind"], ns)[-2:])
p_vl = np.mean(orden(res_T1["vanleer"], ns)[-2:])
VERIFICACION += [
    {"test": "T1 advección upwind: orden ≈ 1", "valor": p_up, "criterio": "|p − 1| < 0.2", "pasa": abs(p_up - 1) < 0.2},
    {"test": "T1 advección van Leer: orden ≈ 2", "valor": p_vl, "criterio": "p > 1.8", "pasa": p_vl > 1.8},
    {"test": "T2 advección–difusión: orden ≈ 2", "valor": p2, "criterio": "p > 1.8", "pasa": p2 > 1.8},
]

**El limitador cuesta poco aquí.** Upwind difumina la onda hasta casi borrarla: es la **difusión numérica**, que reaparecerá en la sección 6. Van Leer la conserva casi intacta.

### T3 — Solución manufacturada 2D con el flujo de vórtices
En 2D no hay solución exacta sencilla con un flujo de vórtices. El **método de las soluciones manufacturadas** le da la vuelta al problema: se elige una solución y se calcula qué fuente $s$ la hace exacta. Elegimos
$$ c^\ast = 1 + a\cos(k_x x)\cos(k_y y)\,e^{-\lambda t}, \qquad k_x = \pi/L_x,\ k_y = \pi/L_y, $$
que cumple la condición de pared ($\partial c/\partial n = 0$). Sustituyendo en la ecuación, con $\nabla\cdot\mathbf{u}=0$:
$$ s = \partial_t c^\ast + \mathbf{u}\cdot\nabla c^\ast - D\nabla^2 c^\ast = -\lambda(c^\ast-1) + u\,\partial_x c^\ast + v\,\partial_y c^\ast + D(k_x^2+k_y^2)(c^\ast-1). $$
Si el código resuelve la ecuación con esa fuente, debe recuperar $c^\ast$ con el orden del esquema. El test ejercita a la vez la advección en 2D, las paredes, la construcción de velocidades y la difusión.

In [ ]:
def caso_mms(nx, ny, Lx=10.0, Ly=4.0, A=15.0, D=0.2, lam=0.1, a=0.5, T=2.0, w=0.5):
    m = Malla(nx, ny, Lx, Ly)
    kx, ky = np.pi / Lx, np.pi / Ly
    norm = A * Ly / np.pi                              # |u| máx ~ A
    Xv, Yv = np.meshgrid(m.xf, m.yf)
    psi = norm * (w * np.sin(kx * Xv) + (1 - w) * np.sin(2 * kx * Xv)) * np.sin(ky * Yv)
    u, v = velocidades_desde_psi(psi, m)
    X, Y = m.X, m.Y
    uc = norm * (w * np.sin(kx * X) + (1 - w) * np.sin(2 * kx * X)) * ky * np.cos(ky * Y)
    vc = -norm * (w * kx * np.cos(kx * X) + (1 - w) * 2 * kx * np.cos(2 * kx * X)) * np.sin(ky * Y)
    exacta = lambda t: 1 + a * np.cos(kx * X) * np.cos(ky * Y) * np.exp(-lam * t)
    def fuente(t):
        e = a * np.exp(-lam * t)
        cx = -e * kx * np.sin(kx * X) * np.cos(ky * Y)
        cy = -e * ky * np.cos(kx * X) * np.sin(ky * Y)
        cm1 = e * np.cos(kx * X) * np.cos(ky * Y)
        return -lam * cm1 + uc * cx + vc * cy + D * (kx ** 2 + ky ** 2) * cm1
    rhs = lambda t, c: operador_transporte(c, u, v, D, m, "vanleer") + fuente(t)
    dt = dt_estable(m, A, D, cfl=0.4); n = int(math.ceil(T / dt)); dt = T / n
    c = exacta(0.0)
    for i in range(n):
        c = paso_ssprk2(c, rhs, i * dt, dt)
    err = c - exacta(T)
    return m, err, float(np.mean(np.abs(err))), float(np.abs(divergencia_discreta(u, v, m)).max())

mallas_mms = [(20, 8), (40, 16), (80, 32), (160, 64)]
out_mms = [caso_mms(nx, ny) for nx, ny in mallas_mms]
e_mms = [o[2] for o in out_mms]
p_mms = orden(e_mms, [nx for nx, _ in mallas_mms])
div_max = max(o[3] for o in out_mms)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={"width_ratios": [1.4, 1]})
m_, err_, _, _ = out_mms[2]
im = a1.imshow(err_, origin="lower", extent=[0, m_.Lx, 0, m_.Ly], cmap="RdBu_r",
               vmin=-np.abs(err_).max(), vmax=np.abs(err_).max(), aspect="equal")
plt.colorbar(im, ax=a1, shrink=0.8, label="error")
a1.set(xlabel="x [cm]", ylabel="y [cm]", title="Error a t = 2 min (malla 80×32)"); a1.grid(False)
nxs = [nx for nx, _ in mallas_mms]
a2.loglog(nxs, e_mms, marker="^", color=COLOR_ESQ["vanleer"], label=f"van Leer (p ≈ {np.mean(p_mms[-2:]):.2f})")
a2.loglog(nxs, e_mms[-1] * (np.array(nxs) / nxs[-1]) ** -2.0, ls=":", color=GRIS_MEDIO, label="pendiente 2")
a2.minorticks_off(); a2.set_xticks(nxs); a2.set_xticklabels([f"{a}×{b}" for a, b in mallas_mms], fontsize=9)
a2.set(xlabel="malla", ylabel="error L1", title="Convergencia de la solución manufacturada"); a2.legend()
fig.suptitle("T3 · Solución manufacturada 2D (advección por vórtices + difusión + paredes)", fontweight="bold")
fig.tight_layout(); plt.show()
print(f"Divergencia discreta máxima del campo de velocidades: {div_max:.1e}")
VERIFICACION += [
    {"test": "T3 solución manufacturada 2D: orden ≈ 2", "valor": float(np.mean(p_mms[-2:])), "criterio": "p > 1.7", "pasa": float(np.mean(p_mms[-2:])) > 1.7},
    {"test": "T3b divergencia discreta nula", "valor": div_max, "criterio": "< 1e-10", "pasa": div_max < 1e-10},
]

### T4 — Conservación con secreción en la pared
En el estómago 2D cerrado con secreción en la pared, cada cantidad total debe cumplir su balance exactamente, salvo redondeo:
- $\int T\,dV - r_{acid}\,t$ es constante.
- $\int B_t\,dV$ es constante.
- $\int (G+E)\,dV - r_G\,t$ es constante.
- $\int (S+P)\,dV$ es constante.

### T5 — Positividad y principio del máximo
Se transporta un bloque discontinuo (0 o 1) con los vórtices alternantes. Un esquema TVD no puede crear valores fuera de [0, 1].

In [ ]:
m4 = Malla(50, 20, 10.0, 4.0)
sim4 = Estomago2D(m4)
H4 = sim4.integrar(estado_bolo(sim4), 10.0, cada=0.5)
V = sim4.vol_celda_mL / 1000
res = {
    "protones T": np.abs(H4.totales(iT) - sim4.q.r_acid * H4.t - H4.totales(iT)[0]) / H4.totales(iT).max(),
    "tampón Bt": np.abs(H4.totales(iB) - H4.totales(iB)[0]) / H4.totales(iB)[0],
    "enzima G+E": np.abs(H4.totales(iG) + H4.totales(iE) - sim4.q.r_G * H4.t - H4.totales(iE)[0]) / (H4.totales(iG) + H4.totales(iE)).max(),
    "proteína S+P": np.abs(H4.totales(iS) + H4.totales(iP) - H4.totales(iS)[0]) / H4.totales(iS)[0],
}
max_res = max(r.max() for r in res.values())

# T5: bloque discontinuo con vórtices alternantes, sin difusión
m5 = Malla(80, 32, 10.0, 4.0)
sim5 = Estomago2D(m5)
c5 = ((m5.X > 2) & (m5.X < 4) & (m5.Y > 1.5) & (m5.Y < 2.5)).astype(float)
dt5 = dt_estable(m5, sim5.pt.U, 0.0); n5 = int(math.ceil(3.0 / dt5)); dt5 = 3.0 / n5
rhs5 = lambda t, c: operador_adveccion(c, *sim5.velocidad(t), m5, "vanleer")
for i in range(n5):
    c5 = paso_ssprk2(c5, rhs5, i * dt5, dt5)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.3]})
for (nombre, r), col in zip(res.items(), ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]):
    a1.semilogy(H4.t, r + 1e-18, color=col, lw=1.4, label=nombre)
a1.set(xlabel="t [min]", ylabel="residuo relativo", title="T4 · Balances (malla 50×20)", ylim=(1e-18, 1)); a1.legend(fontsize=9)
im = a2.imshow(c5, origin="lower", extent=[0, 10, 0, 4], cmap="Greys", vmin=0, vmax=1, aspect="equal")
plt.colorbar(im, ax=a2, shrink=0.75)
a2.set(xlabel="x [cm]", ylabel="y [cm]", title=f"T5 · Bloque tras 3 min: min {c5.min():.1e}, max {c5.max():.6f}"); a2.grid(False)
fig.tight_layout(); plt.show()
VERIFICACION += [
    {"test": "T4 conservación con secreción (4 balances)", "valor": max_res, "criterio": "< 1e-12", "pasa": max_res < 1e-12},
    {"test": "T5 positividad y máximo (bloque 0/1)", "valor": max(-c5.min(), c5.max() - 1),
     "criterio": "fuera de [0,1] < 1e-12", "pasa": c5.min() > -1e-12 and c5.max() < 1 + 1e-12},
]

### T6 — pH vectorizado frente al solucionador escalar de la fase 1b
`resolver_h` resuelve el equilibrio en todas las celdas a la vez con Newton. Para tener la respuesta exacta, construimos estados con un pH "verdadero" conocido, entre 0.5 y 6.5, y calculamos su $T$. Los dos solucionadores, el Newton vectorizado y `brentq` celda a celda, deben recuperar ese pH.

### T7 — Regresión: con mezcla perfecta, el 2D reproduce el 0D
Si el estado inicial es uniforme y la secreción también, el transporte no hace nada: el flujo es exactamente libre de divergencia y los gradientes son nulos. Entonces cada celda debe evolucionar como el modelo 0D. Comprobamos dos cosas:
- La malla 20×8 con vórtices coincide con una sola celda hasta el redondeo.
- La sola celda converge a la solución 0D de referencia (`Radau`, tolerancias estrictas) con **orden 2 en Δt**, el de Heun.

In [ ]:
# Estados con pH conocido entre 0.5 y 6.5: se construye T a partir del pH "verdadero"
rng = np.random.default_rng(0)
pH_true = rng.uniform(0.5, 6.5, 400); Bs = rng.uniform(0, 100, 400)
pKa = ParamQ().pKa
h_true = 10 ** (3 - pH_true)
Ts = h_true + sum((Bs / len(pKa)) * h_true / (h_true + 10 ** (3 - pk)) for pk in pKa)
h_vec = resolver_h(Ts, Bs, pKa)
def F_escalar(x, T, B):
    h = 10 ** (3 - x)
    return h + sum((B / len(pKa)) * h / (h + 10 ** (3 - pk)) for pk in pKa) - T
pH_esc = np.array([brentq(F_escalar, -1, 9, args=(T, B), xtol=1e-14) for T, B in zip(Ts, Bs)])
e_pH = float(max(np.max(np.abs((3 - np.log10(h_vec)) - pH_true)), np.max(np.abs(pH_esc - pH_true))))
print(f"T6: error máximo en pH (Newton vectorizado y brentq frente al pH verdadero) = {e_pH:.1e}")
VERIFICACION.append({"test": "T6 pH vectorizado y escalar vs pH verdadero", "valor": e_pH, "criterio": "< 1e-10", "pasa": e_pH < 1e-10})

# --- T7 ---
pt_unif = ParamT(secrecion="uniforme")
def estado_uniforme(sim):
    Cb = estado_bolo(sim)                               # mismos totales que el bolo...
    return np.broadcast_to(Cb.mean(axis=(1, 2))[:, None, None], Cb.shape).copy()   # ...repartidos uniformemente
sim2d = Estomago2D(Malla(20, 8, 10.0, 4.0), pt=pt_unif)
sim0d = Estomago2D(Malla(1, 1, 10.0, 4.0), pt=pt_unif)
T7 = 20.0
H2 = sim2d.integrar(estado_uniforme(sim2d), T7, dt=0.01, cada=1.0)
H0 = sim0d.integrar(estado_uniforme(sim2d)[:, :1, :1], T7, dt=0.01, cada=1.0)
dif_2d_0d = float(np.max(np.abs(H2.C - H0.C[..., :1, :1]) / (np.abs(H0.C).max(axis=(0, 2, 3))[None, :, None, None] + 1e-30)))

y_ini = estado_uniforme(sim2d)[:, 0, 0]
def rhs_0d(t, y):
    return sim0d.rhs(t, y[:, None, None])[:, 0, 0]
ref0d = solve_ivp(rhs_0d, (0, T7), y_ini, method="Radau", rtol=1e-12, atol=1e-14, dense_output=True)
errs_dt, dts7 = [], [0.04, 0.02, 0.01, 0.005]
for dt in dts7:
    Hd = sim0d.integrar(y_ini[:, None, None], T7, dt=dt, cada=T7)
    yr = ref0d.sol(Hd.t).T
    esc = np.abs(yr).max(axis=0) + 1e-30
    errs_dt.append(float(np.max(np.abs(Hd.C[:, :, 0, 0] - yr) / esc)))
p7 = [math.log2(errs_dt[i] / errs_dt[i + 1]) for i in range(len(dts7) - 1)]
print("Errores 0D vs Radau:", ["%.2e" % e for e in errs_dt], " órdenes:", [round(x, 2) for x in p7])
VERIFICACION += [
    {"test": "T7a 2D uniforme = una celda", "valor": dif_2d_0d, "criterio": "< 1e-10", "pasa": dif_2d_0d < 1e-10},
    {"test": "T7b una celda → 0D Radau: orden ≈ 2 en Δt", "valor": float(np.mean(p7[-2:])), "criterio": "|p − 2| < 0.25", "pasa": abs(np.mean(p7[-2:]) - 2) < 0.25},
]

## 4. Modelo 1D: dispersión axial del ácido

Un modelo 1D promedia cada sección transversal y deja solo la coordenada a lo largo del estómago: del fondo ($x=0$) al píloro ($x = L$). En un tubo cerrado no hay velocidad media neta, así que toda la mezcla se resume en un **coeficiente de dispersión axial** $D_{ax}$. Es el equivalente 1D de $D_{mix}$.

El ácido se secreta en el 60 % proximal (cuerpo gástrico) y la comida está repartida en la zona central. Los diagramas espacio–tiempo muestran cómo avanza el frente ácido para dos valores de $D_{ax}$.

In [ ]:
def caso_1d_estomago(D_ax, nx=100, t_final=60.0):
    m = Malla(nx, 1, 10.0, 4.0)
    sim = Estomago2D(m, pt=ParamT(D_mix=D_ax, flujo="reposo"))
    C0 = estado_bolo(sim, centro=(5.0, 2.0), radio=2.0, delta=0.3)   # en 1D el "bolo" es un tramo
    return sim, sim.integrar(C0, t_final, cada=0.5)

casos_1d = {D: caso_1d_estomago(D) for D in (0.05, 1.0)}
fig, axs = plt.subplots(1, 3, figsize=(13, 4.0), gridspec_kw={"width_ratios": [1, 1, 0.9]})
for a, (D, (sim, H)) in zip(axs[:2], casos_1d.items()):
    im = a.imshow(H.pH[:, 0, :], origin="lower", aspect="auto", cmap=CMAP_PH, vmin=0.8, vmax=6,
                  extent=[0, sim.m.Lx, 0, H.t[-1]])
    a.axvline(sim.pt.x_sec * sim.m.Lx, color="white", lw=1, ls="--")
    a.text(0.3, H.t[-1] * 0.93, "secreción", color="white", fontsize=9)
    a.set(xlabel="x: fondo → píloro [cm]", ylabel="t [min]", title=f"pH(x, t),  D_ax = {D} cm²/min"); a.grid(False)
plt.colorbar(im, ax=axs[:2], shrink=0.85, label="pH")
for D, (sim, H), col in zip(casos_1d, casos_1d.values(), ("#2a78d6", "#eb6834")):
    axs[2].plot(H.t, H.pH_alimento(), color=col, label=f"D_ax = {D}")
axs[2].set(xlabel="t [min]", ylabel="pH medio de la comida", title="pH que 've' la comida"); axs[2].legend()
plt.show()

**Lectura.** El ácido entra en la comida como un **frente** que avanza desde el lado secretor. Delante del frente, la comida sigue tamponada a pH ~6.
- Con dispersión baja, el frente apenas avanza: una hora después, la parte distal de la comida sigue a pH 6, mientras junto a la zona secretora el pH baja de 1. Es una versión 1D de la **bolsa ácida** (*acid pocket*) que se observa clínicamente: ácido sin tamponar en el estómago proximal tras una comida.
- Con dispersión alta, el frente cruza la comida en unos 50 min y luego todo se acidifica.

El **pH que ve la comida**, que es lo que controla la pepsina, depende mucho de un parámetro de mezcla que el modelo 0D no tenía.

## 5. Modelo 2D: un bolo de 50 g de carne

**Escenario.** Losa de 10 × 4 cm con 5 cm de profundidad, es decir, 200 mL: 50 g de carne y 150 mL de agua, como en la fase 1b. El bolo, un disco de 1.5 cm de radio (≈ 35 mL), está en la zona proximal y no se mueve. El ácido se secreta por las paredes superior e inferior del 60 % proximal. Se comparan tres modelos:
1. **0D**: mezcla perfecta, los mismos totales repartidos uniformemente (fase 1b sin vaciado).
2. **2D en reposo**: solo $D_{mix}$.
3. **2D con vórtices**: $D_{mix}$ más el flujo alternante.

In [ ]:
def campo_flujo(sim, t):
    u, v = sim.velocidad(t)
    uc = 0.5 * (u[:, 1:] + u[:, :-1]); vc = 0.5 * (v[1:, :] + v[:-1, :])
    return uc, vc

m_dem = Malla(50, 20, 10.0, 4.0)
sim_dem = Estomago2D(m_dem)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
for a, (t, tit) in zip(axs, [(sim_dem.pt.T_w / 4, "modo 1: un vórtice"), (3 * sim_dem.pt.T_w / 4, "modo 2: dos vórtices")]):
    uc, vc = campo_flujo(sim_dem, t)
    rapidez = np.hypot(uc, vc)
    im = a.imshow(rapidez, origin="lower", extent=[0, 10, 0, 4], cmap="Greys", vmin=0, vmax=sim_dem.pt.U, aspect="equal", alpha=0.6)
    a.streamplot(m_dem.xc, m_dem.yc, uc, vc, color="#2a78d6", density=1.1, linewidth=0.9, arrowsize=0.9)
    a.set(xlabel="x [cm]", ylabel="y [cm]", title=f"{tit}  (t = {t*60:.0f} s)", xlim=(0, 10), ylim=(0, 4)); a.grid(False)
plt.colorbar(im, ax=axs, shrink=0.8, label="|u| [cm/min]")
fig.suptitle(f"Flujo cinemático sustituto: los modos se alternan cada {sim_dem.pt.T_w*60:.0f} s", fontweight="bold")
plt.show()

In [ ]:
T_EXP = 60.0
MALLA_EXP = (50, 20)
t0 = time.time()
RES = {}
for nombre, pt in [("2D con vórtices", ParamT()), ("2D en reposo", ParamT(flujo="reposo"))]:
    sim = Estomago2D(Malla(*MALLA_EXP, 10.0, 4.0), pt=pt)
    RES[nombre] = (sim, sim.integrar(estado_bolo(sim), T_EXP, cada=0.5))
sim0 = Estomago2D(Malla(1, 1, 10.0, 4.0), pt=ParamT(secrecion="uniforme"))
sim_ref = RES["2D con vórtices"][0]
C0_unif = estado_bolo(sim_ref).mean(axis=(1, 2))[:, None, None]
RES["0D (mezcla perfecta)"] = (sim0, sim0.integrar(C0_unif, T_EXP, cada=0.5))
print(f"Simulaciones del experimento: {time.time() - t0:.0f} s")

In [ ]:
instantes = [0, 5, 15, 30, 60]
fig, axs = plt.subplots(2, len(instantes), figsize=(14, 4.6))
for fila, nombre in enumerate(["2D en reposo", "2D con vórtices"]):
    sim, H = RES[nombre]
    for col, ti in enumerate(instantes):
        k = int(np.argmin(np.abs(H.t - ti)))
        a = axs[fila, col]
        im = a.imshow(H.pH[k], origin="lower", extent=[0, 10, 0, 4], cmap=CMAP_PH, vmin=0.8, vmax=6, aspect="equal")
        a.contour(sim.m.X, sim.m.Y, H.C[k, iS], levels=[0.5 * H.C[0, iS].max()], colors="white", linewidths=0.8)
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
        if fila == 0:
            a.set_title(f"t = {ti} min")
        if col == 0:
            a.set_ylabel(nombre)
fig.colorbar(im, ax=axs, shrink=0.8, label="pH")
fig.suptitle("pH en el estómago 2D (contorno blanco: posición de la comida)", fontweight="bold")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.0))
for nombre in ["0D (mezcla perfecta)", "2D en reposo", "2D con vórtices"]:
    sim, H = RES[nombre]
    axs[0].plot(H.t, H.pH_alimento(), color=COLOR_MOD[nombre], label=nombre)
    if nombre != "0D (mezcla perfecta)":
        axs[1].plot(H.t, H.pH_liquido(), color=COLOR_MOD[nombre], label=nombre)
    axs[2].plot(H.t, 100 * H.frac_hidrolizada(14.0), color=COLOR_MOD[nombre], label=nombre)
for a in axs[:2]:
    a.axhline(3, color=GRIS_MEDIO, lw=0.8, ls=":")
axs[0].set(xlabel="t [min]", ylabel="pH", title="pH medio dentro de la comida"); axs[0].legend(fontsize=9)
axs[1].set(xlabel="t [min]", ylabel="pH", title="pH medio del líquido (fuera de la comida)"); axs[1].legend(fontsize=9)
axs[2].set(xlabel="t [min]", ylabel="% de la proteína", title="Proteína hidrolizada")
fig.tight_layout(); plt.show()

def t_cruce(t, y, umbral):
    i = np.where(y < umbral)[0]
    if i.size == 0: return np.nan
    i = i[0]
    return float(t[i - 1] + (umbral - y[i - 1]) * (t[i] - t[i - 1]) / (y[i] - y[i - 1])) if i > 0 else float(t[0])

filas = []
for nombre, (sim, H) in RES.items():
    filas.append({"modelo": nombre, "t hasta pH comida ≤ 3 [min]": t_cruce(H.t, H.pH_alimento(), 3.0),
                  "pH comida a 30 min": float(np.interp(30, H.t, H.pH_alimento())),
                  "pH líquido a 10 min": float(np.interp(10, H.t, H.pH_liquido())) if H.C.shape[-1] > 1 else np.nan,
                  "proteína hidrolizada a 60 min [%]": 100 * float(H.frac_hidrolizada(14.0)[-1]),
                  "pH mínimo (pared) a 30 min": float(np.interp(30, H.t, H.pH.min(axis=(1, 2))))})
TABLA_EXP = pd.DataFrame(filas)
TABLA_EXP.style.format(precision=2).hide(axis="index")

In [ ]:
# Animación del pH (2D con vórtices). Pon ANIMAR = False para ahorrar tiempo.
ANIMAR = True
if ANIMAR:
    sim, H = RES["2D con vórtices"]
    fig, a = plt.subplots(figsize=(7, 3.2))
    im = a.imshow(H.pH[0], origin="lower", extent=[0, 10, 0, 4], cmap=CMAP_PH, vmin=0.8, vmax=6, aspect="equal")
    plt.colorbar(im, ax=a, shrink=0.8, label="pH"); a.grid(False)
    titulo = a.set_title("")
    def frame(k):
        im.set_data(H.pH[k]); titulo.set_text(f"pH, 2D con vórtices — t = {H.t[k]:.1f} min")
        return im, titulo
    anim = animation.FuncAnimation(fig, frame, frames=range(0, len(H.t), 2), interval=120, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

**Lectura del experimento.**
- **La mezcla perfecta es optimista.** En el modelo 0D, todo el ácido secretado titula la comida de inmediato. En 2D, parte del ácido se queda en el **líquido**, que llega a pH 1.5–2 junto a la pared secretora (una "bolsa ácida"), mientras el interior de la comida sigue tamponado. Resultado: la comida tarda más en bajar de pH 3 y, a los 60 min, se ha hidrolizado aproximadamente la mitad de proteína que en 0D. Esa diferencia es una estimación del **error de modelo** que cometía la fase 1b por suponer mezcla perfecta.
- **Los vórtices apenas cambian lo que le pasa a la comida.** Con $D_{mix} = 0.5$ cm²/min el líquido ya está bastante mezclado, y el flujo solo lo acidifica unas décimas más. El pH dentro de la comida casi no cambia, porque el cuello de botella es la **entrada del ácido en el sólido**, no su transporte por el líquido. Los mapas lo muestran: el núcleo del bolo sigue a pH ~6 mientras se acidifica desde fuera, como un núcleo que se encoge (*shrinking core*). El flujo solo deforma ese núcleo.
- **La dinámica de la hidrólisis cambia de forma.** Al principio, el 2D hidroliza algo *más* que el 0D, porque la superficie del bolo se acidifica pronto y allí la pepsina trabaja. Al final hidroliza mucho menos, porque el interior tarda en acidificarse.
- **Conexión con tu idea original.** Esto es justo lo que cambia la **destrucción mecánica**: romper el bolo en partículas multiplica la superficie y acorta la distancia que el ácido debe recorrer. El modelo predice que la mezcla del líquido, por sí sola, no basta para acelerar la digestión química de un bolo compacto.

## 6. Error numérico del experimento y sensibilidad al cierre de mezcla

### 6.1 Convergencia de malla con Richardson
Resolvemos el caso con vórtices en tres mallas (25×10, 50×20 y 100×40), cada una con la mitad de $\Delta x$ que la anterior, hasta 30 min. Con las métricas $q_h$ calculamos:
$$ p_{obs} = \log_2\frac{q_{4h} - q_{2h}}{q_{2h} - q_h}, \qquad q_{exacto} \approx q_h + \frac{q_h - q_{2h}}{2^{p} - 1}, \qquad e_h \approx \left|\frac{q_h - q_{2h}}{2^{p}-1}\right| . $$

In [ ]:
MALLAS_CONV = [(25, 10), (50, 20), (100, 40)]
T_CONV = 30.0
t0 = time.time()
conv = {}
for nx, ny in MALLAS_CONV:
    sim = Estomago2D(Malla(nx, ny, 10.0, 4.0))
    H = sim.integrar(estado_bolo(sim), T_CONV, cada=1.0)
    conv[(nx, ny)] = {"pH comida a 30 min": float(H.pH_alimento()[-1]),
                      "proteína hidrolizada a 30 min [%]": 100 * float(H.frac_hidrolizada(14.0)[-1]),
                      "dispersión del pH a 30 min": float(H.dispersion_pH()[-1])}
print(f"Estudio de malla: {time.time() - t0:.0f} s")

filas = []
for met in conv[MALLAS_CONV[0]]:
    q4, q2, q1 = (conv[mm][met] for mm in MALLAS_CONV)
    p = math.log2(abs(q4 - q2) / abs(q2 - q1)) if abs(q2 - q1) > 0 else np.nan
    p_uso = p if (np.isfinite(p) and 0.5 < p < 4) else 1.0          # si no es asintótico, se usa p = 1 (conservador)
    e_rich = abs(q1 - q2) / (2 ** p_uso - 1)
    filas.append({"métrica": met, "25×10": q4, "50×20": q2, "100×40": q1, "orden observado": p,
                  "extrapolado (Richardson)": q1 + (q1 - q2) / (2 ** p_uso - 1),
                  "error estimado malla 100×40": e_rich,
                  "error estimado malla 50×20": abs(q2 - (q1 + (q1 - q2) / (2 ** p_uso - 1)))})
CONV = pd.DataFrame(filas)
e_pH50 = float(CONV.loc[CONV["métrica"] == "pH comida a 30 min", "error estimado malla 50×20"].iloc[0])
VERIFICACION.append({"test": "T8 error de malla del pH de la comida (50×20)", "valor": e_pH50,
                     "criterio": "< 0.05 unidades de pH", "pasa": e_pH50 < 0.05})
CONV.style.format(precision=4).hide(axis="index")

### 6.2 Sensibilidad al cierre de mezcla $D_{mix}$
Repetimos el caso con vórtices variando $D_{mix}$. El valor más bajo **no está convergido en malla**: su número de Péclet de celda es alto y la difusión numérica compite con la física. Lo incluimos precisamente para mostrar ese efecto.

In [ ]:
D_vals = [0.05, 0.2, 0.5, 2.0]
t0 = time.time()
sens = {}
for D in D_vals:
    if D == 0.5:
        sens[D] = RES["2D con vórtices"][1]; continue
    sim = Estomago2D(Malla(*MALLA_EXP, 10.0, 4.0), pt=ParamT(D_mix=D))
    sens[D] = sim.integrar(estado_bolo(sim), T_EXP, cada=0.5)
print(f"Sensibilidad a D_mix: {time.time() - t0:.0f} s")
dx = 10.0 / MALLA_EXP[0]
colores_D = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.0))
for D, col in zip(D_vals, colores_D):
    Pe = ParamT().U * dx / D
    ls = "--" if Pe > 10 else "-"
    a1.plot(sens[D].t, sens[D].pH_alimento(), color=col, ls=ls, label=f"D_mix = {D}  (Pe celda = {Pe:.0f})")
    a2.plot(sens[D].t, sens[D].dispersion_pH(), color=col, ls=ls)
H0 = RES["0D (mezcla perfecta)"][1]
a1.plot(H0.t, H0.pH_alimento(), color=GRIS, lw=1.2, label="0D (mezcla perfecta)")
a1.set(xlabel="t [min]", ylabel="pH medio de la comida", title="Efecto del cierre de mezcla"); a1.legend(fontsize=8.5)
a2.set(xlabel="t [min]", ylabel="desviación típica del pH", title="Heterogeneidad (discontinua: no convergido)")
fig.tight_layout(); plt.show()

**Qué aprendemos sobre el error.**
- Con $D_{mix} = 0.5$ cm²/min, el **error numérico** de malla en el pH de la comida (tabla de 6.1) es de centésimas de unidad de pH. Es pequeño frente a la diferencia entre los modelos 0D y 2D, que es de medio punto de pH a los 30 min.
- La **incertidumbre de modelo** por el cierre de mezcla es mucho mayor. Con $D_{mix}$ pequeño, el ácido penetra mal en la comida y su pH baja mucho más despacio. Con $D_{mix}$ grande, el resultado se acerca al 0D. Este parámetro, que aquí también representa la porosidad del bolo, es el que la fase 3 (flujo real) y la de partículas (superficie de la comida) deben acotar.
- Con $D_{mix}$ pequeño, el **número de Péclet de celda** es alto y el resultado depende de la malla: parte de la mezcla que se ve es numérica (curvas discontinuas). Para resolver mezcla con poca difusión hay que refinar la malla, con un coste ∝ $1/\Delta x^3$ en 2D explícito, o usar métodos lagrangianos.

**Registro de validación.** Esta fase no añade filas al `registro_validacion.csv`. No tiene vaciado, así que no puede compararse con las métricas de retención ni de pH tras una comida completa. Lo hará la fase 3, al acoplar transporte, flujo y vaciado.

## 7. Resumen de verificación

In [ ]:
ver = pd.DataFrame(VERIFICACION)
ver["resultado"] = np.where(ver["pasa"], "✅ pasa", "❌ FALLA")
display(ver.drop(columns="pasa").style.format({"valor": "{:.3g}"}).hide(axis="index"))
n_ok = int(ver["pasa"].sum())
print(f"\n{n_ok}/{len(ver)} tests superados")
assert n_ok == len(ver), "Hay tests de verificación que no pasan"

## 8. Siguientes pasos

**Establecido en esta fase**
- Tenemos un solucionador de transporte 1D/2D en volúmenes finitos verificado: conserva la masa exactamente, es positivo y tiene orden 2 en espacio y tiempo (solución manufacturada).
- La química de 1b está acoplada celda a celda. Con mezcla perfecta, el modelo se reduce exactamente al 0D.
- La mezcla perfecta sobreestima la digestión de un bolo compacto: a 60 min, el 2D hidroliza aproximadamente la mitad de proteína que el 0D. El cuello de botella es la entrada del ácido en la comida.
- El transporte introduce un parámetro nuevo, la difusividad de mezcla $D_{mix}$, que también gobierna la penetración en el bolo. Es la principal fuente de incertidumbre de esta fase y no podemos medirlo con lo que tenemos.

**Fase 3: el flujo de verdad.**
1. Resolver Navier–Stokes 2D (Lattice Boltzmann o proyección) con paredes que se contraen como ondas peristálticas, y sustituir `Estomago2D.velocidad` por ese campo. El resto del código no cambia.
2. Medir la dispersión efectiva que produce el flujo real, para acotar $D_{mix}$ o eliminarlo si la malla resuelve la mezcla.
3. Más adelante, en la fase de partículas: romper el bolo, porque es lo que más acelera la química según este experimento.
4. Añadir la **salida por el píloro** (vaciado) y volver a evaluar las métricas del registro de validación.